In [19]:
import pandas as pd
import numpy as np
from matplotlib.backend_tools import ToolSetCursor

In [20]:
training_ds = pd.read_csv('data_flat/training_dataset.csv')

In [21]:
training_ds.columns

Index(['route_id', 'station_code', 'date_YYYY_MM_DD', 'departure_time_utc',
       'executor_capacity_cm3', 'route_score', 'stop_id', 'lat', 'lng', 'type',
       'zone_id', 'num_packages', 'total_volume', 'avg_volume',
       'total_service_time', 'avg_outgoing_travel', 'std_outgoing_travel',
       'min_outgoing_travel', 'max_outgoing_travel', 'num_stops',
       'unique_zones', 'executor_capacity_cm3_route', 'station_code_route',
       'actual_sequence', 'sequence_pct', 'invalid_sequence_score'],
      dtype='object')

In [3]:
travel_times = pd.read_csv('data_flat/travel_times.csv')

In [4]:
travel_dict = {}

for _, row in travel_times.iterrows():

    travel_dict[
        (
            row["route_id"],
            row["from_stop"],
            row["to_stop"]
        )
    ] = row["travel_time_seconds"]

# funzione costo

In [5]:
BIG_M = 1e6

def cost(route_id, a, b):

    if a == b:
        return 0

    return travel_dict.get(
        (route_id, a, b),
        BIG_M
    )


In [6]:
BIG_M


1000000.0

In [7]:
high = training_ds[training_ds['route_score']=='High'].copy()

In [8]:
route_id = training_ds["route_id"].iloc[0]

route_df = training_ds[
    training_ds["route_id"] == route_id
].copy()

In [9]:
actual_stops = (
    route_df
    .sort_values("actual_sequence")
    ["stop_id"]
    .tolist()
)

In [10]:
actual_stops

['VE',
 'TG',
 'GP',
 'HT',
 'AG',
 'QM',
 'SF',
 'BY',
 'CW',
 'VW',
 'JH',
 'LB',
 'FH',
 'TK',
 'PT',
 'DL',
 'GU',
 'KN',
 'NR',
 'SC',
 'IJ',
 'XB',
 'KP',
 'IW',
 'TH',
 'MR',
 'CG',
 'LG',
 'DJ',
 'NL',
 'UR',
 'ZE',
 'VA',
 'BA',
 'YH',
 'DQ',
 'CP',
 'DN',
 'UJ',
 'UW',
 'HN',
 'KJ',
 'GN',
 'CA',
 'TC',
 'NE',
 'LD',
 'AF',
 'KA',
 'BT',
 'MW',
 'KM',
 'UI',
 'BG',
 'CO',
 'KG',
 'EO',
 'IA',
 'YE',
 'RG',
 'VC',
 'BZ',
 'ZU',
 'KU',
 'GS',
 'NU',
 'MA',
 'BP',
 'IM',
 'EY',
 'MQ',
 'RA',
 'US',
 'PJ',
 'GW',
 'RY',
 'FY',
 'YR',
 'WS',
 'NM',
 'HO',
 'GB',
 'PS',
 'LY',
 'LK',
 'QE',
 'CK',
 'HG',
 'FF',
 'MO',
 'YN',
 'TQ',
 'JM',
 'HW',
 'IP',
 'SI',
 'CM',
 'HB',
 'XD',
 'SQ',
 'ZP',
 'EH',
 'YJ',
 'TY',
 'EX',
 'AD',
 'EC',
 'QO',
 'UN',
 'BE',
 'PB',
 'HR',
 'ZB',
 'WJ',
 'UU',
 'SD',
 'YY',
 'PX',
 'QX']

In [11]:
def edge_accuracy_stops(
    pred,
    true
):

    pred_edges = set(
        zip(pred[:-1], pred[1:])
    )

    true_edges = set(
        zip(true[:-1], true[1:])
    )

    return (
        len(pred_edges & true_edges)
        / len(true_edges)
    )

In [13]:
route_df.columns

Index(['route_id', 'station_code', 'date_YYYY_MM_DD', 'departure_time_utc',
       'executor_capacity_cm3', 'route_score', 'stop_id', 'lat', 'lng', 'type',
       'zone_id', 'num_packages', 'total_volume', 'avg_volume',
       'total_service_time', 'avg_outgoing_travel', 'std_outgoing_travel',
       'min_outgoing_travel', 'max_outgoing_travel', 'num_stops',
       'unique_zones', 'executor_capacity_cm3_route', 'station_code_route',
       'actual_sequence', 'sequence_pct', 'invalid_sequence_score'],
      dtype='object')

In [14]:
route_df["type"].value_counts(dropna=False)


type
Dropoff    118
Station      1
Name: count, dtype: int64

In [15]:
#recupero stop della station
station_stop = route_df.loc[
    route_df["type"] == "Station",
    "stop_id"
].iloc[0]

station_stop

'VE'

In [35]:
def zone_otsp_paper(
        route_id,
        zone_stops,
        start_stop,
        next_zone_anchor_stop=None
):

    remaining = set(zone_stops)

    current = start_stop

    tour = []

    while remaining:

        nxt = min(
            remaining,
            key=lambda s:
                cost(
                    route_id,
                    current,
                    s
                )
        )

        tour.append(nxt)

        remaining.remove(nxt)

        current = nxt

    if next_zone_anchor_stop is not None:
        tour.append(next_zone_anchor_stop)

    return tour

# test

In [17]:
def build_stop_sequence_paper(
    route_df,
    predicted_zones
):

    route_id = route_df["route_id"].iloc[0]

    station_stop = route_df.loc[
        route_df["type"] == "Station",
        "stop_id"
    ].iloc[0]

    final_sequence = []

    # la prima zona parte dalla station
    previous_stop = station_stop

    for i, zone in enumerate(predicted_zones):

        current_stops = route_df.loc[
          (route_df["zone_id"] == zone)
           &
          (route_df["type"] != "Station"),
            "stop_id"
         ].tolist()

        if len(current_stops) == 0:
            continue

        # next zone anchor
        if i < len(predicted_zones) - 1:

            next_zone = predicted_zones[i + 1]

            next_stops = route_df.loc[
                route_df["zone_id"] == next_zone,
                "stop_id"
            ].tolist()

            anchor = next_zone_anchor(
                         route_df,
                        next_zone
                )

        else:

            anchor = None

        tour = zone_otsp_paper(
            route_id,
            current_stops,
            previous_stop,
            anchor
        )

        if anchor is not None:

            zone_seq = tour[:-1]

            previous_stop = zone_seq[-1]

        else:

            zone_seq = tour

            previous_stop = tour[-1]

        for s in zone_seq:

            if s not in final_sequence:
                final_sequence.append(s)

    return final_sequence

### best_stop = stop della zona successiva
### con costo medio minimo rispetto agli stop della zona corrente

In [18]:
def next_zone_anchor(
    route_df,
    next_zone
):

    next_zone_df = route_df[
        route_df["zone_id"] == next_zone
    ].copy()

    if len(next_zone_df) == 0:
        return None

    center_lat = next_zone_df["lat"].mean()
    center_lng = next_zone_df["lng"].mean()

    next_zone_df["dist_to_center"] = np.sqrt(
        (next_zone_df["lat"] - center_lat) ** 2
        +
        (next_zone_df["lng"] - center_lng) ** 2
    )

    anchor_stop = next_zone_df.loc[
        next_zone_df["dist_to_center"].idxmin(),
        "stop_id"
    ]

    return anchor_stop

In [ ]:
next_zone = predicted_zones[1]

anchor = next_zone_anchor(
    route_df,
    next_zone
)

print(anchor)

In [ ]:
route_df.loc[
    route_df["stop_id"] == anchor,
    ["stop_id", "zone_id"]
]